In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/Users/fuyuxuan/Downloads/testout_1.3.csv")
A = df.to_numpy(dtype=float)
std = np.sqrt(np.diag(A))
corr = A / np.outer(std, std)

def projection_psd(M: np.ndarray) -> np.ndarray:
    M = (M + M.T) / 2.0
    vals, vecs = np.linalg.eigh(M)
    vals = np.clip(vals, 0.0, None)
    return vecs @ np.diag(vals) @ vecs.T

def higham_near_corr(C: np.ndarray, tol: float = 1e-14, max_iter: int = 10000) -> np.ndarray:
    Y = corr.copy()
    delta_S = np.zeros_like(corr)
    for _ in range(max_iter):
        R = Y - delta_S
        X = projection_psd(R)
        delta_S = X - R

        Y_new = X.copy()
        np.fill_diagonal(Y_new, 1.0)

        if np.linalg.norm(Y_new - Y, ord="fro") < tol:
            Y = Y_new
            break
        Y = Y_new
        
    return (Y + Y.T) / 2.0

C_high = higham_near_corr(corr)
cov_high = C_high * np.outer(std, std)

out = pd.DataFrame(cov_high, columns=df.columns)
print(out)

         x1        x2        x3        x4        x5
0  1.173986 -0.623870 -0.294335 -0.057677 -0.693888
1 -0.623870  1.318197  0.016449  0.448579  0.143703
2 -0.294335  0.016449  0.918102  0.354067  0.246866
3 -0.057677  0.448579  0.354067  0.894764 -0.217062
4 -0.693888  0.143703  0.246866 -0.217062  0.522607
